# 面试问题：MoE 的 Expert Parallel、capacity、All-to-All 与负载均衡怎样实现？

**回答主线。** 稀疏 MoE 先由 router 为每个 token 选择 top-k expert，再按 expert 所在 rank 重排 token，执行 All-to-All、专家 FFN 和反向 All-to-All，最后按 gate 权重合并。总参数可以很大，但每个 token 只激活少数专家；真正瓶颈常是热点 expert、跨节点通信和尾部 rank。

下面从零实现路由、容量裁剪、dispatch 矩阵、专家前向、Switch-style 辅助损失与 loss-free bias 更新。教学代码在单进程模拟 rank，不把 `all_to_all` 框架调用当作答案。


In [ ]:
import hashlib, json, math
from dataclasses import dataclass
import numpy as np

# 固定 logits 和 expert 权重，方便比较不同均衡策略。
rng153 = np.random.default_rng(153)

def softmax153(x):
    x = np.asarray(x, dtype=np.float64)
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)

p153 = softmax153(np.array([[1.0, 2.0, 3.0]]))
assert p153.shape == (1, 3)
assert np.allclose(p153.sum(axis=1), 1.0)
assert np.argmax(p153) == 2


## 1. Router 输出 top-k expert 与原始 gate 权重

Top-k 用于离散 dispatch，softmax 概率用于合并。路由 bias 可以参与“选谁”，但不一定参与最终 gate 值；两者混用会让负载控制直接扭曲专家输出尺度。


In [ ]:
def topk_route153(logits, k, routing_bias=None):
    # bias 只影响排名；返回的 gate 仍来自原始 logits。
    logits = np.asarray(logits, dtype=np.float64)
    biased = logits if routing_bias is None else logits + routing_bias[None, :]
    experts = np.argsort(-biased, axis=1, kind="stable")[:, :k]
    original_prob = softmax153(logits)
    gates = np.take_along_axis(original_prob, experts, axis=1)
    gates = gates / gates.sum(axis=1, keepdims=True)
    return experts, gates, original_prob

logits153 = rng153.normal(size=(12, 4)); logits153[:, 0] += 2.0
experts153, gates153, probs153 = topk_route153(logits153, k=2)
assert experts153.shape == gates153.shape == (12, 2)
assert np.allclose(gates153.sum(axis=1), 1.0)
assert np.all((experts153 >= 0) & (experts153 < 4))


## 2. Capacity 限制每个 expert 接收的 token 数

常用容量是 `ceil(capacity_factor * T * k / E)`。超额 token 可以丢弃、转第二选择或排队；任何策略都应稳定排序并报告 drop rate。这里按 gate 从高到低接收，分数相同时按 token/slot 保证确定性。


In [ ]:
def apply_capacity153(experts, gates, num_experts, capacity_factor):
    # 对每个 expert 独立选择最高 gate，返回与 top-k 表同形的 accepted mask。
    tokens, k = experts.shape
    capacity = math.ceil(capacity_factor * tokens * k / num_experts)
    accepted = np.zeros_like(experts, dtype=bool)
    for expert in range(num_experts):
        candidates = [(float(gates[t, j]), t, j) for t in range(tokens) for j in range(k) if experts[t, j] == expert]
        candidates.sort(key=lambda item: (-item[0], item[1], item[2]))
        for _, token, slot in candidates[:capacity]:
            accepted[token, slot] = True
    return accepted, capacity

accepted153, capacity153 = apply_capacity153(experts153, gates153, 4, capacity_factor=0.75)
loads153 = np.bincount(experts153[accepted153], minlength=4)
assert capacity153 == math.ceil(0.75 * 12 * 2 / 4)
assert np.all(loads153 <= capacity153)
assert accepted153.sum() <= experts153.size


## 3. Dispatch 矩阵揭示 All-to-All 热点

假设 token 初始按 data rank 分布、expert 按 `expert % world_size` 放置，发送矩阵的 `(src,dst)` 就是需要搬运的 token-choice 数。总量平衡不代表链路平衡，还要看每行每列和跨节点边。


In [ ]:
def dispatch_matrix153(experts, accepted, world_size):
    # token 的初始 rank 与 expert rank 都显式计算，便于拓扑分析。
    matrix = np.zeros((world_size, world_size), dtype=np.int64)
    for token, slot in zip(*np.where(accepted)):
        src = token % world_size
        dst = int(experts[token, slot]) % world_size
        matrix[src, dst] += 1
    return matrix

traffic153 = dispatch_matrix153(experts153, accepted153, world_size=2)
assert traffic153.shape == (2, 2)
assert traffic153.sum() == accepted153.sum()
assert np.all(traffic153 >= 0)


## 4. 专家前向与 gate 合并必须恢复 token 顺序

Dispatch 会把 token 按 expert 重排；combine 阶段必须 scatter 回原 token，并只在已接收 choice 上重新归一化。下面用独立两层 FFN 模拟专家，显式执行 gather—expert—scatter。


In [ ]:
hidden153, expert_hidden153, num_experts153 = 5, 7, 4
w1_153 = rng153.normal(scale=0.2, size=(num_experts153, hidden153, expert_hidden153))
w2_153 = rng153.normal(scale=0.2, size=(num_experts153, expert_hidden153, hidden153))

def expert_forward153(x, expert):
    # tanh 只作教学非线性，核心是 expert-specific 两层参数。
    return np.tanh(x @ w1_153[expert]) @ w2_153[expert]

def moe_forward153(x, experts, gates, accepted):
    output = np.zeros_like(x, dtype=np.float64)
    weight_sum = np.zeros(len(x), dtype=np.float64)
    for token, slot in zip(*np.where(accepted)):
        expert, weight = int(experts[token, slot]), float(gates[token, slot])
        output[token] += weight * expert_forward153(x[token], expert)
        weight_sum[token] += weight
    valid = weight_sum > 0
    output[valid] /= weight_sum[valid, None]
    return output, valid

xmoe153 = rng153.normal(size=(12, hidden153))
ymoe153, valid153 = moe_forward153(xmoe153, experts153, gates153, accepted153)
assert ymoe153.shape == xmoe153.shape
assert np.isfinite(ymoe153).all()
assert np.all(ymoe153[~valid153] == 0.0)


## 5. 负载指标同时看 CV、最大/平均和 drop

平均负载由 token 数决定，无法发现 straggler。最大/平均直接对应最慢 rank 的尾部；变异系数衡量整体离散；drop rate 则暴露容量不足。指标必须按 expert、rank、数据 slice 分层。


In [ ]:
def load_metrics153(experts, accepted, num_experts):
    # 未接收 choice 计入 drop，但不计入实际 expert load。
    load = np.bincount(experts[accepted], minlength=num_experts).astype(np.float64)
    mean = max(float(load.mean()), 1e-12)
    return {"load": load, "cv": float(load.std() / mean), "max_over_mean": float(load.max() / mean), "drop_rate": float(1 - accepted.mean())}

load_report153 = load_metrics153(experts153, accepted153, 4)
assert len(load_report153["load"]) == 4
assert load_report153["max_over_mean"] >= 1.0
assert 0.0 <= load_report153["drop_rate"] <= 1.0


## 6. Switch-style 辅助损失把路由频率与平均概率对齐

Top-1 情况下常见形式是 `E * sum(f_i * p_i)`，其中 `f_i` 是离散 token 比例，`p_i` 是平均 router 概率。它可导地推动均衡，但系数太大会与语言建模目标竞争。


In [ ]:
def switch_aux_loss153(logits):
    # f 来自 top-1 计数，p 来自未裁剪 softmax 概率。
    prob = softmax153(logits)
    top1 = np.argmax(prob, axis=1)
    experts = prob.shape[1]
    frequency = np.bincount(top1, minlength=experts) / len(top1)
    mean_prob = prob.mean(axis=0)
    return float(experts * np.sum(frequency * mean_prob)), frequency, mean_prob

uniform_logits153 = np.tile(np.eye(4), (3, 1)) * 8.0
aux153, freq153, meanp153 = switch_aux_loss153(uniform_logits153)
collapsed_aux153, collapsed_freq153, _ = switch_aux_loss153(logits153)
assert np.allclose(freq153, 0.25)
assert abs(aux153 - 1.0) < 1e-3
assert collapsed_freq153[0] > freq153[0] and collapsed_aux153 > 0


## 7. Auxiliary-loss-free bias 用反馈控制路由排名

一类方法监控实际负载：过载 expert 降低 routing bias，欠载 expert 提高 bias。Bias 只影响 top-k 选择，合并 gate 仍取原始 affinity。步长过大会振荡，因此 bias、负载窗口和更新速率都属于 checkpoint 状态。


In [ ]:
def loss_free_update153(logits, bias, k=1, step_size=0.05):
    # 目标是每个 expert 接收相同 choice 数，按偏差方向更新 bias。
    chosen, _, _ = topk_route153(logits, k, bias)
    load = np.bincount(chosen.ravel(), minlength=logits.shape[1]).astype(np.float64)
    target = load.sum() / len(load)
    new_bias = bias + step_size * np.sign(target - load)
    return new_bias, load

bias153 = np.zeros(4)
initial_bias153, initial_load153 = loss_free_update153(logits153, bias153, step_size=0.1)
bias153 = initial_bias153
for _ in range(30):
    bias153, final_load153 = loss_free_update153(logits153, bias153, step_size=0.1)
assert initial_bias153[0] < 0
assert np.any(initial_bias153[1:] > 0)
assert final_load153.max() <= initial_load153.max()


## 8. 发布制品绑定 placement、router 与容量策略

同一模型权重在不同 expert placement 下会产生不同通信热点。恢复时必须同时加载 router bias、expert 到 rank 的映射、capacity/drop 策略和拓扑版本，并以端到端 token 输出做 smoke test。


In [ ]:
def placement_artifact153(model_revision, placement, router_bias, capacity_factor):
    # 排序后的 canonical JSON 防止字典顺序制造虚假版本变化。
    payload = json.dumps({"model": model_revision, "placement": placement, "bias": list(map(float, router_bias)), "capacity_factor": capacity_factor}, sort_keys=True, separators=(",", ":"))
    return payload, hashlib.sha256(payload.encode()).hexdigest()

placement153 = {str(e): e % 2 for e in range(4)}
payload153, digest153 = placement_artifact153("moe-v4", placement153, bias153, 0.75)
assert len(digest153) == 64
assert json.loads(payload153)["placement"]["3"] == 1
assert json.loads(payload153)["model"] == "moe-v4"


## 面试总结

- Router 做 top-k，dispatch 做 token 重排与 All-to-All，expert 执行 FFN，combine 恢复 token 顺序。
- Capacity 防止热点撑爆 buffer，但必须报告 drop、reroute 与质量代价。
- 辅助损失通过训练目标均衡；loss-free bias 通过反馈改变路由排名，两者都需要稳定性与 checkpoint 设计。
- Expert Parallel 的有效性能由最慢 rank、跨节点流量和 placement 决定，不是只看激活参数量。

延伸阅读：[Switch Transformers](https://arxiv.org/abs/2101.03961)、[GShard](https://arxiv.org/abs/2006.16668)、[DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437)。
